# Decode trajectory viewer

Load one JSONL record from `python -m relay.dump_decode_trajectories` (field `trajectory`: list of token-id vectors per step). Use the slider to scrub through decoding; optionally export a GIF with `matplotlib.animation`.

For Sudoku / n-queens, `relay.viz_puzzle_state` maps ids to a small grid for `imshow`.

In [ ]:
from pathlib import Path
import json
import sys

# Ensure repo root is on path (notebook may run with cwd = plotting_scripts/ or project root)
_here = Path.cwd()
if (_here / "relay" / "viz_puzzle_state.py").is_file():
    _root = _here
elif (_here.parent / "relay" / "viz_puzzle_state.py").is_file():
    _root = _here.parent
else:
    _root = _here  # last resort; may still fail if cwd is elsewhere
if str(_root) not in sys.path:
    sys.path.insert(0, str(_root))

import matplotlib.pyplot as plt
import numpy as np
import ipywidgets as w
from IPython.display import display

from relay.viz_puzzle_state import (
    sudoku_ids_to_grid,
    n_queens_ids_to_grid,
    infer_dataset,
)

JSONL = Path("../decode_trajectories.jsonl")  # set path to your dump
LINE_INDEX = 0  # which example line to visualize

lines = JSONL.read_text().strip().splitlines()
rec = json.loads(lines[LINE_INDEX])
traj = rec["trajectory"]
tags = rec.get("tags") or {}
ds = infer_dataset(tags)
print("dataset:", ds, "steps:", len(traj))

In [ ]:
def ids_to_grid(step_ids):
    if "sudoku" in ds.lower():
        return sudoku_ids_to_grid(step_ids)
    if "queens" in ds.lower() or "n_queens" in ds.lower():
        return n_queens_ids_to_grid(step_ids)
    return np.array(step_ids, dtype=np.float32)

fig, ax = plt.subplots(figsize=(4, 4))
im = ax.imshow(ids_to_grid(traj[0]), vmin=-1, vmax=10, cmap="viridis")
ax.set_title(f"step 0 / {len(traj)-1}")
plt.colorbar(im, ax=ax)

def update(step):
    step = int(step)
    g = ids_to_grid(traj[step])
    im.set_data(g)
    ax.set_title(f"step {step} / {len(traj)-1}")
    fig.canvas.draw_idle()

slider = w.IntSlider(value=0, min=0, max=len(traj) - 1, step=1, description="step")
w.interact(update, step=slider)
plt.show()